In [1]:
import requests
import folium
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
base_url = "http://openapi.seoul.go.kr:8088/7661686a586b696e3335467873514b/json/bikeList/1/5/"
response = requests.get(base_url)  

response    #<Response [200]> = 정상출력

<Response [200]>

In [3]:
json_data = response.json()
json_data

# {'rentBikeStatus': {'list_total_count': 5,
#   'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다.'},
#   'row': [{'rackTotCnt': '15',
#     'stationName': '102. 망원역 1번출구 앞',
#     'parkingBikeTotCnt': '10',
#     'shared': '67',
#     'stationLatitude': '37.55564880',
#     'stationLongitude': '126.91062927',
#     'stationId': 'ST-4'},
#    {'rackTotCnt': '14',
#     'stationName': '103. 망원역 2번출구 앞',
#     'parkingBikeTotCnt': '20',
#     'shared': '143',
#     'stationLatitude': '37.55495071',
#     'stationLongitude': '126.91083527',
#     'stationId': 'ST-5'},
#    {'rackTotCnt': '13',
#     'stationName': '104. 합정역 1번출구 앞',
#     'parkingBikeTotCnt': '18',
#     'shared': '138',
#     'stationLatitude': '37.55073929',
#     'stationLongitude': '126.91508484',
#     'stationId': 'ST-6'},
#    {'rackTotCnt': '5',
#     'stationName': '105. 합정역 5번출구 앞',
# ...
#     'parkingBikeTotCnt': '1',
#     'shared': '8',
#     'stationLatitude': '37.54864502',
#     'stationLongitude': '126.91282654',
#     'stationId': 'ST-8'}]}}
# Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

{'rentBikeStatus': {'list_total_count': 5,
  'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다.'},
  'row': [{'rackTotCnt': '15',
    'stationName': '102. 망원역 1번출구 앞',
    'parkingBikeTotCnt': '14',
    'shared': '93',
    'stationLatitude': '37.55564880',
    'stationLongitude': '126.91062927',
    'stationId': 'ST-4'},
   {'rackTotCnt': '14',
    'stationName': '103. 망원역 2번출구 앞',
    'parkingBikeTotCnt': '8',
    'shared': '57',
    'stationLatitude': '37.55495071',
    'stationLongitude': '126.91083527',
    'stationId': 'ST-5'},
   {'rackTotCnt': '13',
    'stationName': '104. 합정역 1번출구 앞',
    'parkingBikeTotCnt': '3',
    'shared': '23',
    'stationLatitude': '37.55073929',
    'stationLongitude': '126.91508484',
    'stationId': 'ST-6'},
   {'rackTotCnt': '5',
    'stationName': '105. 합정역 5번출구 앞',
    'parkingBikeTotCnt': '0',
    'shared': '0',
    'stationLatitude': '37.55000687',
    'stationLongitude': '126.91482544',
    'stationId': 'ST-7'},
   {'rackTotCnt': '12',
   

In [4]:
json_data.get("rentBikeStatus",{}).get("RESULT", {}).get("CODE", "")

'INFO-000'

In [ ]:
def fetch_bike_data():
    base_url = "http://openapi.seoul.go.kr:8088/7661686a586b696e3335467873514b/json/bikeList/"
    start = 1
    end = 1000
    step = 1000
    data_frames= []
    while True:
        url = f'{base_url}{start}/{end}/'  #url세팅
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Status Code:  {response.status_code}")
            break
        json_data = response.json()
        try:
            rent_bike_status = json_data['rentBikeStatus']
            result_code = rent_bike_status['RESULT']['CODE']
        except KeyError:
            print('json 오류')
            break
        if result_code == "INFO-200":
            print("데이터 없음")
            break
        elif result_code == "INFO-000":
            print(f'시작:{start}, 끝: {end}')
            try:
                bike_data = rent_bike_status['row']
                if bike_data:
                    df = pd.DataFrame(bike_data)
                    data_frames.append(df)
            except KeyError:
                print('데이터 없음')
        elif result_code== "ERROR-336":
            print("데이터 요청은 1000건을 넘길수 없습니다")
            break
        else:
            print(f"알 수 없는 오류: {result_code}")
            break
        start += step
        end += step
    if data_frames:
        final_df = pd.concat(data_frames, ignore_index = True)      #인덱스를 제외하고 데이터프레임을 붙여줘
        return final_df
df = fetch_bike_data()
df

시작:1, 끝: 1000
시작:1001, 끝: 2000
시작:2001, 끝: 3000
json 오류


,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,15,100,37.55564880,126.91062927,ST-4
1,14,103. 망원역 2번출구 앞,9,64,37.55495071,126.91083527,ST-5
2,13,104. 합정역 1번출구 앞,2,15,37.55073929,126.91508484,ST-6
3,5,105. 합정역 5번출구 앞,0,0,37.55000687,126.91482544,ST-7
4,12,106. 합정역 7번출구 앞,1,8,37.54864502,126.91282654,ST-8
...,...,...,...,...,...,...,...
2743,32,6184. 한강버스 마곡 선착장,52,163,37.57402039,126.84357452,ST-2492
2744,8,6185.가양나들목,10,125,37.57341003,126.84345245,ST-3418
2745,12,6187.마곡119안전센터 맞은편,29,242,37.55534744,126.82072449,ST-3415
2746,10,6188.금호아파트,17,170,37.55619049,126.86463928,ST-3419


In [14]:
'''
rackTotCnt	거치대개수	
parkingBikeTotCnt	자전거주차총건수	
shared	거치율	
stationLatitude	위도	
stationLongitude	경도	
stationId	대여소ID	
stationName	대여소이름	
'''


'\nrackTotCnt\t거치대개수\t\nparkingBikeTotCnt\t자전거주차총건수\t\nshared\t거치율\t\nstationLatitude\t위도\t\nstationLongitude\t경도\t\nstationId\t대여소ID\t\nstationName\t대여소이름\t\n'

In [27]:
df['parkingBikeTotCnt'] = df['parkingBikeTotCnt'].astype(int)

df = df[df.parkingBikeTotCnt > 0]

In [15]:
df['stationLatitude'] = df['stationLatitude'].astype(float)
df['stationLongitude'] = df['stationLongitude'].astype(float)

In [28]:
df

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,15,100,37.555649,126.910629,ST-4
1,14,103. 망원역 2번출구 앞,9,64,37.554951,126.910835,ST-5
2,13,104. 합정역 1번출구 앞,2,15,37.550739,126.915085,ST-6
4,12,106. 합정역 7번출구 앞,1,8,37.548645,126.912827,ST-8
5,10,107. 수협 연남동지점,2,20,37.557846,126.918716,ST-9
...,...,...,...,...,...,...,...
2743,32,6184. 한강버스 마곡 선착장,52,163,37.574020,126.843575,ST-2492
2744,8,6185.가양나들목,10,125,37.573410,126.843452,ST-3418
2745,12,6187.마곡119안전센터 맞은편,29,242,37.555347,126.820724,ST-3415
2746,10,6188.금호아파트,17,170,37.556190,126.864639,ST-3419


,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,15,100,37.555649,126.910629,ST-4
1,14,103. 망원역 2번출구 앞,9,64,37.554951,126.910835,ST-5
2,13,104. 합정역 1번출구 앞,2,15,37.550739,126.915085,ST-6
4,12,106. 합정역 7번출구 앞,1,8,37.548645,126.912827,ST-8
5,10,107. 수협 연남동지점,2,20,37.557846,126.918716,ST-9
...,...,...,...,...,...,...,...
2743,32,6184. 한강버스 마곡 선착장,52,163,37.574020,126.843575,ST-2492
2744,8,6185.가양나들목,10,125,37.573410,126.843452,ST-3418
2745,12,6187.마곡119안전센터 맞은편,29,242,37.555347,126.820724,ST-3415
2746,10,6188.금호아파트,17,170,37.556190,126.864639,ST-3419


In [ ]:
bike_map = folium.Map(location=[df['stationLatitude'].mean(), 
                                df['stationLongitude'].mean()],
                                zoom_start = 12)
for index,data in df.iterrows():
    popup_str = '{} 자전거주차총건수: {}대'.format(
        data['stationName'], data['parkingBikeTotCnt']
    )
    popup = folium.Popup(popup_str, max_width=600)
    folium.Marker(location = [data['stationLatitude'],
                              data['stationLongitude']],
                              popup=popup).add_to(bike_map)
bike_map